In [6]:
import os
from google.colab import drive

# It's safer to ensure the mount point is truly empty or doesn't exist
# For robust removal of non-empty directories, use shutil.rmtree
import shutil
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')

# Now, create an empty directory for mounting and mount drive
os.makedirs('/content/drive', exist_ok=True)
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
"""
Support Vector Machine (SVM) - MNIST Classification
AIGC 5102 - Final Project: Handwritten Digit Classification

This script trains an SVM classifier on the PCA-reduced MNIST dataset
(95% variance retained) and performs systematic hyperparameter tuning
via Grid Search using an 80/20 stratified train/validation split.

The primary tuning objective is Macro F1-score. The script also reports
Macro Precision and Macro Recall.

Pipeline:
    1. Mount Google Drive (Colab).
    2. Load PCA-reduced train and test CSVs.
    3. Stratified 80/20 train/validation split (random_state=0).
    4. Grid Search using PredefinedSplit so tuning happens on the
       validation set rather than a generic k-fold CV.
    5. Refit best params on train-only, evaluate on validation set.
    6. Refit best params on full training set (train+val), evaluate on test.
    7. Save best hyperparameters, classification reports, and confusion
       matrix to the SVM output folder.

Rationale:
    - SVM with RBF kernel is highly sensitive to feature count, so we
      use the 95% variance PCA data (~150 components) to keep training
      tractable on Colab's runtime.
    - All random states are set to 0 per the project rubric.

Run in Google Colab:
    from google.colab import drive
    drive.mount('/content/drive')
    !python "/content/drive/MyDrive/AI Integration & Governance/Into to ML/Final Project/Model Training/SVM/train_svm.py"
"""

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

# If running in Google Colab, mount Drive first:
#   from google.colab import drive
#   drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/AI Integration & Governance/Into to ML/Final Project"

# Input: PCA-reduced (95% variance) training and test data
PCA_DIR = f"{BASE_DIR}/Preprocessing/PCA File Output"
TRAIN_FILE = f"{PCA_DIR}/mnist_train_pca_95variance.csv"
TEST_FILE = f"{PCA_DIR}/mnist_test_pca_95variance.csv"

# Output: folder for SVM artifacts
OUTPUT_DIR = f"{BASE_DIR}/Model Training/SVM"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BEST_PARAMS_FILE = f"{OUTPUT_DIR}/svm_best_params.json"
VAL_REPORT_FILE = f"{OUTPUT_DIR}/svm_validation_report.txt"
TEST_REPORT_FILE = f"{OUTPUT_DIR}/svm_test_report.txt"
CONFUSION_MATRIX_FILE = f"{OUTPUT_DIR}/svm_test_confusion_matrix.png"
GRID_RESULTS_FILE = f"{OUTPUT_DIR}/svm_grid_search_results.csv"
SUMMARY_FILE = f"{OUTPUT_DIR}/svm_summary.txt"


# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
def load_csv(path):
    """Load a PCA-reduced MNIST CSV and return features and labels."""
    print(f"Loading {path} ...")
    df = pd.read_csv(path)
    labels = df.iloc[:, 0].astype(int).values
    features = df.iloc[:, 1:].astype(np.float32).values
    print(f"  -> shape: {df.shape}")
    return features, labels


X_train_full, y_train_full = load_csv(TRAIN_FILE)
X_test, y_test = load_csv(TEST_FILE)

print(f"\nTraining set: {X_train_full.shape}, labels: {y_train_full.shape}")
print(f"Test set:     {X_test.shape}, labels: {y_test.shape}")


# ---------------------------------------------------------------------------
# Stratified 80/20 train/validation split
# ---------------------------------------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)
print(f"\nAfter 80/20 stratified split:")
print(f"  Train:      {X_tr.shape}")
print(f"  Validation: {X_val.shape}")


# ---------------------------------------------------------------------------
# Grid Search (tuning on the validation set via PredefinedSplit)
# ---------------------------------------------------------------------------
# Combine train and val into one array, then use PredefinedSplit so that
# GridSearchCV always trains on the 48k train rows and evaluates on the
# 12k validation rows - exactly matching the rubric's "Grid Search on the
# validation set" requirement.
X_combined = np.vstack([X_tr, X_val])
y_combined = np.concatenate([y_tr, y_val])
# -1 marks training samples; 0 marks validation samples.
test_fold = np.array([-1] * len(X_tr) + [0] * len(X_val))
ps = PredefinedSplit(test_fold=test_fold)

# Multi-metric scoring so we can report all three, while refitting on F1.
scoring = {
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
}

param_grid = {
    "C": [1, 10, 100],
    "gamma": ["scale", 0.01, 0.001],
    "kernel": ["rbf"],
}

base_model = SVC(
    random_state=RANDOM_STATE,
    cache_size=1000,        # Larger kernel cache for faster training
    decision_function_shape="ovr",
)

print("\nStarting GridSearchCV ...")
print(f"  param_grid: {param_grid}")
print(f"  scoring:    {list(scoring.keys())}")
print(f"  refit:      f1_macro (primary tuning objective)")

start = time.time()
grid = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=ps,
    scoring=scoring,
    refit=False,        # We'll retrain the best model manually for clarity
    n_jobs=-1,
    verbose=2,
)
grid.fit(X_combined, y_combined)
elapsed = time.time() - start
print(f"\nGrid search complete in {elapsed/60:.2f} minutes.")


# ---------------------------------------------------------------------------
# Extract best hyperparameters (optimized for macro F1)
# ---------------------------------------------------------------------------
cv_results = pd.DataFrame(grid.cv_results_)
best_idx = cv_results["mean_test_f1_macro"].idxmax()
best_params = cv_results.loc[best_idx, "params"]
best_val_f1 = cv_results.loc[best_idx, "mean_test_f1_macro"]
best_val_precision = cv_results.loc[best_idx, "mean_test_precision_macro"]
best_val_recall = cv_results.loc[best_idx, "mean_test_recall_macro"]

print("\n" + "=" * 60)
print("BEST HYPERPARAMETERS (optimized for Macro F1)")
print("=" * 60)
print(json.dumps(best_params, indent=2, default=str))
print(f"\nValidation Macro Precision: {best_val_precision:.6f}")
print(f"Validation Macro Recall:    {best_val_recall:.6f}")
print(f"Validation Macro F1:        {best_val_f1:.6f}")

# Persist best parameters to disk
with open(BEST_PARAMS_FILE, "w") as f:
    json.dump(
        {
            "best_params": {k: (v if isinstance(v, (int, float, str)) else str(v))
                            for k, v in best_params.items()},
            "validation_macro_precision": float(best_val_precision),
            "validation_macro_recall": float(best_val_recall),
            "validation_macro_f1": float(best_val_f1),
            "grid_search_time_minutes": elapsed / 60,
        },
        f,
        indent=2,
    )
cv_results.to_csv(GRID_RESULTS_FILE, index=False)


# ---------------------------------------------------------------------------
# Evaluate on validation set (train on X_tr only)
# ---------------------------------------------------------------------------
print("\nTraining best model on X_tr only for validation metrics ...")
val_model = SVC(
    random_state=RANDOM_STATE,
    cache_size=1000,
    decision_function_shape="ovr",
    **best_params,
)
val_model.fit(X_tr, y_tr)
y_val_pred = val_model.predict(X_val)

val_precision = precision_score(y_val, y_val_pred, average="macro")
val_recall = recall_score(y_val, y_val_pred, average="macro")
val_f1 = f1_score(y_val, y_val_pred, average="macro")
val_accuracy = accuracy_score(y_val, y_val_pred)
val_classification = classification_report(y_val, y_val_pred, digits=4)

print(f"\nValidation Macro Precision: {val_precision:.6f}")
print(f"Validation Macro Recall:    {val_recall:.6f}")
print(f"Validation Macro F1:        {val_f1:.6f}")
print(f"Validation Accuracy:        {val_accuracy:.6f}")

with open(VAL_REPORT_FILE, "w") as f:
    f.write("SVM - Validation Set Results\n")
    f.write("=" * 60 + "\n")
    f.write(f"Best params: {best_params}\n\n")
    f.write(f"Macro Precision: {val_precision:.6f}\n")
    f.write(f"Macro Recall:    {val_recall:.6f}\n")
    f.write(f"Macro F1:        {val_f1:.6f}\n")
    f.write(f"Accuracy:        {val_accuracy:.6f}\n\n")
    f.write("Per-class classification report:\n")
    f.write(val_classification)


# ---------------------------------------------------------------------------
# Evaluate on test set (train on X_train_full = X_tr + X_val)
# ---------------------------------------------------------------------------
print("\nTraining final model on full training set for test metrics ...")
final_model = SVC(
    random_state=RANDOM_STATE,
    cache_size=1000,
    decision_function_shape="ovr",
    **best_params,
)
final_model.fit(X_train_full, y_train_full)
y_test_pred = final_model.predict(X_test)

test_precision = precision_score(y_test, y_test_pred, average="macro")
test_recall = recall_score(y_test, y_test_pred, average="macro")
test_f1 = f1_score(y_test, y_test_pred, average="macro")
test_accuracy = accuracy_score(y_test, y_test_pred)
test_classification = classification_report(y_test, y_test_pred, digits=4)

print(f"\nTest Macro Precision: {test_precision:.6f}")
print(f"Test Macro Recall:    {test_recall:.6f}")
print(f"Test Macro F1:        {test_f1:.6f}")
print(f"Test Accuracy:        {test_accuracy:.6f}")

with open(TEST_REPORT_FILE, "w") as f:
    f.write("SVM - Test Set Results\n")
    f.write("=" * 60 + "\n")
    f.write(f"Best params: {best_params}\n\n")
    f.write(f"Macro Precision: {test_precision:.6f}\n")
    f.write(f"Macro Recall:    {test_recall:.6f}\n")
    f.write(f"Macro F1:        {test_f1:.6f}\n")
    f.write(f"Accuracy:        {test_accuracy:.6f}\n\n")
    f.write("Per-class classification report:\n")
    f.write(test_classification)


# ---------------------------------------------------------------------------
# Confusion matrix (test set)
# ---------------------------------------------------------------------------
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
disp.plot(ax=ax, cmap="Blues", colorbar=True, values_format="d")
ax.set_title("SVM - Test Set Confusion Matrix")
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_FILE, dpi=150)
plt.close(fig)
print(f"\nConfusion matrix saved to: {CONFUSION_MATRIX_FILE}")


# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------
summary = [
    "SVM Training Summary",
    "=" * 60,
    f"Input data:              PCA 95% variance ({X_train_full.shape[1]} features)",
    f"Training samples:        {X_tr.shape[0]}",
    f"Validation samples:      {X_val.shape[0]}",
    f"Test samples:            {X_test.shape[0]}",
    f"Grid search time:        {elapsed/60:.2f} minutes",
    "",
    f"BEST HYPERPARAMETERS (optimized for Macro F1):",
    f"  {best_params}",
    "",
    f"Validation Macro Precision: {val_precision:.6f}",
    f"Validation Macro Recall:    {val_recall:.6f}",
    f"Validation Macro F1:        {val_f1:.6f}",
    f"Validation Accuracy:        {val_accuracy:.6f}",
    "",
    f"Test Macro Precision:       {test_precision:.6f}",
    f"Test Macro Recall:          {test_recall:.6f}",
    f"Test Macro F1:              {test_f1:.6f}",
    f"Test Accuracy:              {test_accuracy:.6f}",
]

summary_text = "\n".join(summary)
print("\n" + summary_text)

with open(SUMMARY_FILE, "w") as f:
    f.write(summary_text)

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print("Done.")


Loading /content/drive/MyDrive/AI Integration & Governance/Into to ML/Final Project/Preprocessing/PCA File Output/mnist_train_pca_95variance.csv ...
  -> shape: (60000, 155)
Loading /content/drive/MyDrive/AI Integration & Governance/Into to ML/Final Project/Preprocessing/PCA File Output/mnist_test_pca_95variance.csv ...
  -> shape: (10000, 155)

Training set: (60000, 154), labels: (60000,)
Test set:     (10000, 154), labels: (10000,)

After 80/20 stratified split:
  Train:      (48000, 154)
  Validation: (12000, 154)

Starting GridSearchCV ...
  param_grid: {'C': [1, 10, 100], 'gamma': ['scale', 0.01, 0.001], 'kernel': ['rbf']}
  scoring:    ['precision_macro', 'recall_macro', 'f1_macro']
  refit:      f1_macro (primary tuning objective)
Fitting 1 folds for each of 9 candidates, totalling 9 fits

Grid search complete in 13.60 minutes.

BEST HYPERPARAMETERS (optimized for Macro F1)
{
  "C": 10,
  "gamma": "scale",
  "kernel": "rbf"
}

Validation Macro Precision: 0.984951
Validation Macr